In [1]:
# Dépendances du notebook
%pip install openpyxl==3.1.3 pandas==3.0.3 s3fs==2026.3.0 -q

Note: you may need to restart the kernel to use updated packages.


# Import des packages nécessaires

In [65]:
import pandas as pd
import pprint
import io
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.chart import BarChart, PieChart, Reference
from openpyxl.chart.label import DataLabelList
from openpyxl.chart.series import DataPoint
from openpyxl.formatting.rule import DataBarRule
from openpyxl.drawing.spreadsheet_drawing import SpreadsheetDrawing

# Import des données
Les données sont disponibles sur le site [DATA AMELI](https://data.ameli.fr/pages/home/).




J’ai choisi de travailler avec deux jeux de données totalement distincts :
**Une vue régionale et départementale**, décrivant les patients selon la pathologie, le traitement chronique ou l’épisode de soins, le sexe, la classe d’âge, la région et le département.  
  - Source : [Effectifs – Patients par pathologie](https://data.ameli.fr/explore/dataset/effectifs/information/)

**Une vue nationale**, portant sur les dépenses remboursées par pathologie et par patient en moyenne.  
  - Source : [Dépenses remboursées par pathologie](https://data.ameli.fr/explore/dataset/depenses/information/)

Comme le fichier **effectifs** est particulièrement volumineux, j’ai opté pour une **lecture en chunks**, ce qui permet de filtrer les données au fur et à mesure du chargement et d’éviter une surcharge mémoire.

J’ai ensuite enrichi ces données en les fusionnant avec un second fichier **region_dept** contenant les libellés des régions et des départements, afin d’obtenir un rendu plus harmonieux et plus lisible dans les visualisations.

Pour garantir un chargement fiable et éviter les problèmes liés aux chemins locaux, j’ai préféré déposer mes fichiers CSV sur Onyxia et les récupérer directement via leur URL MinIO. Cette approche assure un accès stable et reproductible aux données, quel que soit l’environnement d’exécution.

### Effectif de patients par pathologie, sexe, classe d'âge et territoire (département, région)

In [3]:
chunks = []

for chunk in pd.read_csv('https://minio.lab.sspcloud.fr/aissataa/PROJET_MEYTE/effectifs.csv', sep=';', chunksize=100_000,low_memory=False):
    filtered = chunk[(chunk['annee'] >= 2022) & (chunk['dept'] != '999')]
    chunks.append(filtered)

df_effectifs = pd.concat(chunks, ignore_index=True)

#Le datasaet qui va servir de jointure pour récuperer les libelles des départements et région
df_regions_dept = pd.read_csv(
    "https://minio.lab.sspcloud.fr/aissataa/PROJET_MEYTE/LibelleRegionDept.csv",
    sep=";",
    encoding="latin1",
    usecols=["departement", "libelle_departement", "libelle_region"]
)

# Harmonisation des colonnes , les 1 devient des 01 
df_regions_dept["departement"] = df_regions_dept["departement"].astype(str).str.zfill(2)
df_effectifs["dept"] = df_effectifs["dept"].astype(str).str.zfill(2)

# Jointure des 2 :
df_fusion = pd.merge(
    df_effectifs, 
    df_regions_dept, 
    left_on="dept", 
    right_on="departement", 
    how="inner"
)



Ici, je n’utilise pas `.copy()` car mon objectif n’est pas de dupliquer le DataFrame, mais simplement de le renommer.  
En faisant :

In [4]:
df_effectifs = df_fusion
del df_fusion # Et par la suite je supprime le dataFrame pour libérer de l'espace

## Nettoyage du dataset : df_effectifs

### Vérification et traitement des doublons

J’affiche ici les lignes du DataFrame qui apparaissent en doublon afin d’identifier
les répétitions éventuelles dans les données. L’option `keep=False` permet de visualiser
toutes les occurrences d’un doublon, ce qui facilite le diagnostic avant nettoyage.

Le fichier des correspondances *région–département* contient plusieurs doublons,ce qui génère des lignes dupliquées après la jointure.
Pour éviter ces répétitions dans le DataFrame final, j’ai décidé de supprimer les doublons avant de poursuivre l’analyse.


In [5]:
df_effectifs[df_effectifs.duplicated(keep=False)]


,annee,patho_niv1,patho_niv2,patho_niv3,top,cla_age_5,sexe,region,dept,Ntop,Npop,prev,Niveau prioritaire,libelle_classe_age,libelle_sexe,tri,libelle_region,departement,libelle_departement
0,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,NaN,8460,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,59,Nord
1,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,NaN,8460,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,59,Nord
2,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,NaN,8460,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,59,Nord
3,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,NaN,8460,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,59,Nord
4,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,NaN,8460,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,59,Nord
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4878295,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,MCV_APE_IND,45-49,2,75,64,40.0,21590,0.185,"2,3",de 45 à 49 ans,femmes,27.0,Nouvelle-Aquitaine,64,Pyrénées-Atlantiques
4878296,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,MCV_APE_IND,45-49,2,75,64,40.0,21590,0.185,"2,3",de 45 à 49 ans,femmes,27.0,Nouvelle-Aquitaine,64,Pyrénées-Atlantiques
4878297,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,MCV_APE_IND,45-49,2,75,64,40.0,21590,0.185,"2,3",de 45 à 49 ans,femmes,27.0,Nouvelle-Aquitaine,64,Pyrénées-Atlantiques
4878298,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,MCV_APE_IND,45-49,2,75,64,40.0,21590,0.185,"2,3",de 45 à 49 ans,femmes,27.0,Nouvelle-Aquitaine,64,Pyrénées-Atlantiques


#### Nombre de doublons par colonne

In [6]:
df_effectifs.apply(lambda col: col.duplicated().sum())


annee                  4878298
patho_niv1             4878282
patho_niv2             4878251
patho_niv3             4878238
top                    4878221
cla_age_5              4878279
sexe                   4878297
region                 4878282
dept                   4878199
Ntop                   4869980
Npop                   4872749
prev                   4834295
Niveau prioritaire     4878294
libelle_classe_age     4878279
libelle_sexe           4878297
tri                    4878221
libelle_region         4878282
departement            4878199
libelle_departement    4878199
dtype: int64

### Poids du dataset avant et après suppression des doublons

Avant de nettoyer les données, j’affiche le poids mémoire du DataFrame afin d’évaluer l’impact des doublons sur la taille totale du fichier.  
Après suppression des lignes dupliquées, j’affiche à nouveau le poids du dataset pour mesurer le gain en mémoire et vérifier que le nettoyage a bien été appliqué.


In [7]:
buf = io.StringIO()
df_effectifs.info(buf=buf, verbose=False)
print("Poids AVANT nettoyage :", buf.getvalue().splitlines()[-1])



Poids AVANT nettoyage : memory usage: 707.2 MB


In [8]:
# Suppression des doublons
df_effectifs = df_effectifs.drop_duplicates()


In [9]:
# Afficher uniquement la ligne "memory usage"
buf = io.StringIO()
df_effectifs.info(buf=buf, verbose=False)
print("Poids APRÈS nettoyage :", buf.getvalue().splitlines()[-1])

Poids APRÈS nettoyage : memory usage: 141.4 MB


In [10]:
df_effectifs.head()


,annee,patho_niv1,patho_niv2,patho_niv3,top,cla_age_5,sexe,region,dept,Ntop,Npop,prev,Niveau prioritaire,libelle_classe_age,libelle_sexe,tri,libelle_region,departement,libelle_departement
0,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,NaN,8460,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,59,Nord
5,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,60,NaN,2470,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,60,Oise
10,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,62,NaN,5010,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,62,Pas-de-Calais
15,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,80,NaN,2220,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,80,Somme
20,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,44,10,NaN,1570,NaN,3,plus de 95 ans,tous sexes,78.0,Grand Est,10,Aube


In [11]:
# Valeurs manquantes
(
    df_effectifs
    .isna()
    .sum(axis=0)
)

annee                       0
patho_niv1                  0
patho_niv2             101808
patho_niv3             220584
top                         0
cla_age_5                   0
sexe                        0
region                      0
dept                        0
Ntop                   259584
Npop                        0
prev                   259584
Niveau prioritaire      12726
libelle_classe_age          0
libelle_sexe                0
tri                     12726
libelle_region              0
departement                 0
libelle_departement         0
dtype: int64

Dans ce jeu de données, les valeurs indiquées comme NaN ne correspondent pas à de véritables valeurs manquantes. Elles apparaissent simplement lorsqu’un département ne présente aucun cas pour une pathologie donnée. Dans ce contexte, il est donc logique de remplacer ces NaN par 0, afin de refléter correctement l’absence de cas observés.

In [12]:
df_effectifs = df_effectifs.fillna(0)


###  Après avoir passer les NAN à 0

In [13]:
# Valeurs manquantes
(
    df_effectifs
    .isna()
    .sum(axis=0)
)

annee                  0
patho_niv1             0
patho_niv2             0
patho_niv3             0
top                    0
cla_age_5              0
sexe                   0
region                 0
dept                   0
Ntop                   0
Npop                   0
prev                   0
Niveau prioritaire     0
libelle_classe_age     0
libelle_sexe           0
tri                    0
libelle_region         0
departement            0
libelle_departement    0
dtype: int64

### Nettoyage et préparation des données

- J’ai décidé de supprimer les parenthèses dans les variables *patho_niv2* et *patho_niv3*, car elles entraînaient un affichage peu lisible dans les visualisations et ajoutaient trop d’informations inutiles.
- Je me suis concentrée sur la France hexagonale, en incluant la Corse et les DOM.
- Les codes **FRANCE** et **Tout département** correspondent à des agrégats régionaux ou nationaux (valeurs non rattachées à un département).  
  Comme ils faussent l’analyse territoriale, j’ai choisi de les supprimer.


In [14]:
# Nettoyage des parenthèses
cols = ["patho_niv2", "patho_niv3"]
for c in cols:
    df_effectifs.loc[:, c] = df_effectifs[c].str.replace(r"\s*\(.*\)", "", regex=True)

# Grand filtre de nettoyage , je veux gardé que la france hexagonale
hors_hexa = ['Tout département', 'Guadeloupe ', 'Haute-Corse', 'Martinique', 'La Réunion', 'Guyane', 'Mayotte', 'Corse-du-Sud','FRANCE']

df_effectifs = df_effectifs[
    (~df_effectifs['libelle_departement'].astype(str).isin(hors_hexa)) &
    
    # On enlève la région "France" qui englobe tout
    (df_effectifs['libelle_region'].astype(str) != 'FRANCE')
]


Pour ne conserver que les observations réellement exploitables, je filtre le dataset
en gardant uniquement les lignes où la variable `prev` est strictement supérieure à 0.
Cela permet d’éliminer les enregistrements sans prévalence et d’alléger les analyses
et visualisations suivantes.

In [15]:
df_effectifs = df_effectifs[df_effectifs["prev"] > 0]


In [16]:
df_effectifs.dtypes


annee                    int64
patho_niv1                 str
patho_niv2              object
patho_niv3              object
top                        str
cla_age_5                  str
sexe                     int64
region                   int64
dept                       str
Ntop                   float64
Npop                     int64
prev                   float64
Niveau prioritaire      object
libelle_classe_age         str
libelle_sexe               str
tri                    float64
libelle_region             str
departement                str
libelle_departement        str
dtype: object

### Nettoyage des colonnes `patho_niv2` et `patho_niv3` après remplacement des `NaN`

Lors du remplacement des valeurs manquantes (`NaN`) par `0`, les colonnes `patho_niv2` et
`patho_niv3` ont changé de type : elles sont passées du type *string* au type *object*.
Lorsqu’une colonne contient un mélange de valeurs c'est ce que pandas fait numériques, textuelles ou des `NaN`, elle est automatiquement convertie en `object`.
Comme les `NaN` avaient été remplacés par `0`, ces valeurs sont restées sous forme de `"0"` après conversion en `str`.  
Je les ai supprimé pour ne conserver que les niveaux de pathologie valides.


In [17]:
df_effectifs["patho_niv2"] = df_effectifs["patho_niv2"].astype(str).str.strip()
df_effectifs["patho_niv3"] = df_effectifs["patho_niv3"].astype(str).str.strip()


In [18]:
df_effectifs.dtypes


annee                    int64
patho_niv1                 str
patho_niv2                 str
patho_niv3                 str
top                        str
cla_age_5                  str
sexe                     int64
region                   int64
dept                       str
Ntop                   float64
Npop                     int64
prev                   float64
Niveau prioritaire      object
libelle_classe_age         str
libelle_sexe               str
tri                    float64
libelle_region             str
departement                str
libelle_departement        str
dtype: object

In [19]:
df_effectifs = df_effectifs[
    (df_effectifs["patho_niv1"] != "0") &
    (df_effectifs["patho_niv2"] != "0") &
    (df_effectifs["patho_niv3"] != "0")
]

### Suppression des colonnes non utilisées

Dans cette étape, je retire les colonnes qui ne sont pas nécessaires pour l’analyse finale.  
Il s’agit principalement de variables techniques, de codes intermédiaires ou de niveaux d’agrégation
qui ne sont pas exploités dans les visualisations (ex. : codes de tri, niveaux prioritaires, variables
de regroupement, ou encore des informations redondantes comme la colonne **region** issue d’un des
datasets utilisés pour la jointure).

L’option `errors='ignore'` permet d’éviter une erreur si certaines colonnes ont déjà été supprimées
lors d’étapes précédentes du nettoyage.


In [20]:
# Suppression des colonnes inutiles pour le nettoyage final
df_effectifs = df_effectifs.drop(
    columns=[
        "Code département",
        "tri",
        "Niveau prioritaire",
        "region",
        "dept",
        "departement",
        "cla_age_5",
        "top",
        "sexe",

    ],
    errors='ignore' # Évite de faire une erreur si ça a déjà été supprimée
)




### Renommage de certaines colonnes

Pour améliorer la lisibilité du dataset et faciliter la compréhension des analyses,
j’ai choisi de renommer plusieurs colonnes.  
Afin d’obtenir des intitulés plus courts, plus explicites et cohérents avec les visualisations produites par la suite.



In [21]:
# Renommage des colonens
df_effectifs = df_effectifs.rename(columns={
    "libelle_departement": "Département",
    "libelle_region": "Région",
    "prev" : "Prévalence",
    "Npop": "Population de référence",
    "Ntop": "Effectif",
    "libelle_sexe" : "Sexe",
    "libelle_classe_age" : "Classe d'age"


})

In [22]:
df_effectifs = df_effectifs.dropna()

In [23]:
df_effectifs

,annee,patho_niv1,patho_niv2,patho_niv3,Effectif,Population de référence,Prévalence,Classe d'age,Sexe,Région,Département
150,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,150.0,981490,0.015,tous âges,hommes,Ile-de-France,Paris
155,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,30.0,204050,0.015,tous âges,hommes,Centre-Val de Loire,Eure-et-Loir
160,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,20.0,123680,0.016,tous âges,hommes,Bourgogne-Franche-Comté,Jura
170,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,40.0,259740,0.014,tous âges,hommes,Bourgogne-Franche-Comté,Saône-et-Loire
175,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,10.0,65260,0.018,tous âges,hommes,Bourgogne-Franche-Comté,Territoire de Belfort
...,...,...,...,...,...,...,...,...,...,...,...
4878265,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,40.0,47150,0.091,de 45 à 49 ans,femmes,Pays de la Loire,Loire-Atlantique
4878270,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,40.0,21550,0.162,de 45 à 49 ans,femmes,Pays de la Loire,Vendée
4878275,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,30.0,17860,0.174,de 45 à 49 ans,femmes,Bretagne,Côtes-d'Armor
4878280,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,40.0,34710,0.107,de 45 à 49 ans,femmes,Bretagne,Ille-et-Vilaine


### Exploration des niveaux de pathologies

Avant d’appliquer des filtres plus précis, j’affiche les valeurs uniques de `patho_niv2` et `patho_niv3`
(en excluant les valeurs `NaN`). Cela permet de vérifier la cohérence des libellés et d’identifier
éventuels regroupements ou corrections à effectuer avant l’analyse.


In [24]:
print(df_effectifs['patho_niv1'].dropna().unique())
print(df_effectifs['patho_niv2'].dropna().unique())
print(df_effectifs['patho_niv3'].dropna().unique())


<StringArray>
[                                                               'Maladies inflammatoires ou rares ou infection VIH',
                                                                                           'Maladies neurologiques',
                                                                                          'Maladies psychiatriques',
 'Pas de pathologie repérée, traitement, maternité, hospitalisation ou traitement antalgique ou anti-inflammatoire',
                                                                                   'Total consommants tous régimes',
                                                              'Traitements du risque vasculaire (hors pathologies)',
                                                                      'Traitements psychotropes (hors pathologies)',
                                                                                  'Maladies cardioneurovasculaires',
                                                  

# 2 ème dataset que je vais étudier :

# Dépenses remboursées affectées à chaque pathologie

Les données présentent des informations sur les dépenses remboursées par l’ensemble des régimes d’assurance maladie et affectées à chaque pathologie, traitement chronique ou épisode de soins. Ces dépenses sont réparties en trois grandes catégories :

* les soins de ville (consultations médicales, soins infirmiers, actes de kinésithérapie, médicaments, biologie, transports, etc.)  ; 

* les hospitalisations dans les établissements de santé publics ou privés ;

* les prestations en espèces, notamment les indemnités journalières.

## Deux types d’indicateurs sont proposés pour chacune de ces catégories : 

* les dépenses totales annuelles remboursées, qui mesurent le poids financier global d’une pathologie ;

* les dépenses moyennes annuelles remboursées par patient, qui permettent d’apprécier le coût moyen associé à chaque pathologie.

Ces deux indicateurs offrent ainsi une vision complémentaire des dépenses de santé, en combinant à la fois l’impact global et la charge moyenne par patient.

In [25]:
chunks = []

for chunk in pd.read_csv(
    'https://minio.lab.sspcloud.fr/aissataa/PROJET_MEYTE/depenses.csv',
    sep=';',
    chunksize=100_000,
    low_memory=False
):
    filtered = chunk[
        (chunk['annee'] >= 2022) &
        (chunk['montant'] > 0) 
    ]
    chunks.append(filtered)

df_depenses = pd.concat(chunks, ignore_index=True)
df_depenses


,annee,patho_niv1,patho_niv2,patho_niv3,top,dep_niv_1,dep_niv_2,montant,Ntop,N_recourant_au_poste,montant_moy,Niveau prioritaire,tri,type_somme
0,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Hospitalisations (tous secteurs),Hospitalisations liste en sus MCO secteur priv...,3856836,1714310.0,34840.0,2.0,"1,2,3",16.0,Partiel
1,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Hospitalisations (tous secteurs),Total hospitalisations (tous secteurs) rembour...,308350458,1714310.0,1232630.0,180.0,"1,2,3",16.0,Total
2,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Prestations en espèces,Indemnités journalières maladie et AT/MP rembo...,117367790,1714310.0,125760.0,68.0,"1,2,3",16.0,Partiel
3,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Prestations en espèces,Prestations d'invalidité remboursées,263517389,1714310.0,68580.0,154.0,"1,2,3",16.0,Partiel
4,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Soins de ville,Autres dépenses de soins de ville remboursés,9004352,1714310.0,424990.0,5.0,"1,2,3",16.0,Partiel
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4265,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),TPS_NLE_EXC,Soins de ville,Autres produits de santé remboursés,10878069,335640.0,212200.0,32.0,"2,3",55.0,Partiel
4266,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),TPS_NLE_EXC,Soins de ville,Soins de généralistes remboursés,11874138,335640.0,300910.0,35.0,"2,3",55.0,Partiel
4267,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),TPS_NLE_EXC,Soins de ville,Soins dentaires remboursés,6030821,335640.0,118450.0,18.0,"2,3",55.0,Partiel
4268,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),TPS_NLE_EXC,Soins de ville,Total soins de ville remboursés,199465569,335640.0,335630.0,594.0,"2,3",55.0,Total


In [26]:
# Renommage des colonens
df_depenses = df_depenses.rename(columns={
    "Ntop": "Effectif",
    "dep_niv_1": "poste de dépense",
    "dep_niv_2": "sous poste"
})



# Les champs sont les suivants avant nettoyage:

In [27]:
df_depenses.columns


Index(['annee', 'patho_niv1', 'patho_niv2', 'patho_niv3', 'top',
       'poste de dépense', 'sous poste', 'montant', 'Effectif',
       'N_recourant_au_poste', 'montant_moy', 'Niveau prioritaire', 'tri',
       'type_somme'],
      dtype='str')

In [28]:
# Suppression des colonnes inutiles pour le nettoyage final
df_depenses = df_depenses.drop(
    columns=[
        "tri",
        "Niveau prioritaire",
        "type_somme",
        "top",

    ],
    errors='ignore' # Évite de faire une erreur si ça a déjà été supprimée
)




# Nettoyage des données

In [29]:
(
    df_depenses
    .isna()
    .sum(axis=0)
)

annee                     0
patho_niv1                0
patho_niv2              457
patho_niv3              997
poste de dépense          0
sous poste                0
montant                   0
Effectif                 28
N_recourant_au_poste     28
montant_moy              28
dtype: int64

In [30]:
df_depenses = df_depenses.drop_duplicates()


In [31]:
df_depenses


,annee,patho_niv1,patho_niv2,patho_niv3,poste de dépense,sous poste,montant,Effectif,N_recourant_au_poste,montant_moy
0,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Hospitalisations (tous secteurs),Hospitalisations liste en sus MCO secteur priv...,3856836,1714310.0,34840.0,2.0
1,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Hospitalisations (tous secteurs),Total hospitalisations (tous secteurs) rembour...,308350458,1714310.0,1232630.0,180.0
2,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Prestations en espèces,Indemnités journalières maladie et AT/MP rembo...,117367790,1714310.0,125760.0,68.0
3,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Prestations en espèces,Prestations d'invalidité remboursées,263517389,1714310.0,68580.0,154.0
4,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Soins de ville,Autres dépenses de soins de ville remboursés,9004352,1714310.0,424990.0,5.0
...,...,...,...,...,...,...,...,...,...,...
4265,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),Soins de ville,Autres produits de santé remboursés,10878069,335640.0,212200.0,32.0
4266,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),Soins de ville,Soins de généralistes remboursés,11874138,335640.0,300910.0,35.0
4267,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),Soins de ville,Soins dentaires remboursés,6030821,335640.0,118450.0,18.0
4268,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),Soins de ville,Total soins de ville remboursés,199465569,335640.0,335630.0,594.0


# Filtrage avant creation du dashboard

In [32]:
df_depenses = df_depenses.drop('Total consommants tous régimes', errors='ignore')

# Création du fichier Excel

Je vais créer un nouveau fichier Excel qui contiendra les données nettoyées dans un onglet "cleanedData_depenses" et "cleanedData_Pathos"

Pour les dépenses

In [49]:
output_dep = '../DATA/Dashboard_Depenses.xlsx'

with pd.ExcelWriter(output_dep, engine='openpyxl') as writer:
    df_depenses.to_excel(writer, sheet_name='cleanedData_depenses', index=False)

print(f"Fichier dépenses créé : {output_dep}")


Fichier dépenses créé : ../DATA/Dashboard_Depenses.xlsx


Pour les pathologies :

In [82]:
output_eff = '../DATA/Dashboard_Effectifs.xlsx'

with pd.ExcelWriter(output_eff, engine='openpyxl') as writer:
    df_effectifs.to_excel(writer, sheet_name='cleanedData_Effectifs', index=False)

print(f"Fichier effectifs créé : {output_eff}")


Exception ignored in: <function ZipFile.__del__ at 0x7f64bd213880>
Traceback (most recent call last):
  File "/opt/python/lib/python3.13/zipfile/__init__.py", line 2001, in __del__
    self.close()
  File "/opt/python/lib/python3.13/zipfile/__init__.py", line 2019, in close
    self._write_end_record()
  File "/opt/python/lib/python3.13/zipfile/__init__.py", line 2113, in _write_end_record
    self.fp.write(endrec)
  File "/opt/python/lib/python3.13/zipfile/__init__.py", line 867, in write
    n = self.fp.write(data)
AttributeError: 'Workbook' object has no attribute 'write'


Fichier effectifs créé : ../DATA/Dashboard_Effectifs.xlsx


In [83]:
wb_eff = load_workbook(output_eff)

In [84]:
wb_dep = load_workbook(output_dep) 

Calcul des listes _ len_dict

In [85]:
annees = sorted(df_depenses['annee'].dropna().unique())
pathos = sorted(df_depenses['patho_niv1'].dropna().unique())
postes = sorted(df_depenses['poste de dépense'].dropna().unique())

# LISTES POUR FILTRES EFFECTIFS
depts = sorted(df_effectifs['Département'].dropna().unique())
classes_age = sorted(df_effectifs['Classe d\'age'].dropna().unique())
sexes = sorted(df_effectifs['Sexe'].dropna().unique())
regions = sorted(df_effectifs['Région'].dropna().unique())
pathos_eff = sorted(df_effectifs['patho_niv1'].dropna().unique())

# LEN_DICT
len_dict = {
    'len_annee': len(annees) + 1,
    'len_patho_niv1': len(pathos) + 1,
    'len_poste de dépense': len(postes) + 1,
    'len_departement_eff': len(depts) + 1,
    'len_classes_age': len(classes_age) + 1,
    'len_sexe': len(sexes) + 1,
    'len_region': len(regions) + 1,
    'len_patho_effectif': len(pathos_eff) + 1, 
}

pprint.pprint(len_dict)

{'len_annee': 3,
 'len_classes_age': 22,
 'len_departement_eff': 96,
 'len_patho_effectif': 19,
 'len_patho_niv1': 20,
 'len_poste de dépense': 5,
 'len_region': 14,
 'len_sexe': 4}


In [86]:
cols_to_calculate = ['annee', 'patho_niv1', 'poste de dépense', 'sous poste']

cols_to_calculate_effectifs = {
    'annee_eff': 'annee',
    'patho_effectif': 'patho_niv1',
    'age': "Classe d'age",
    'sexe': 'Sexe',
    'region': 'Région',
    'departement_eff': 'Département'
}
len_dict = {}

# BOUCLE POUR DÉPENSES
for col in cols_to_calculate:
    if col == 'patho_niv1':
        # Exclure "Total"
        uniques = df_depenses.loc[
            ~df_depenses[col].isin(['Total', 'Total consommants tous régimes']),
            col
        ].dropna().unique()
    else:
        uniques = df_depenses[col].dropna().unique()
    
    len_dict[f"len_{col}"] = len(uniques) + 1

# BOUCLE POUR EFFECTIFS
for key_name, col_real_name in cols_to_calculate_effectifs.items():
    if col_real_name in df_effectifs.columns:
        if key_name == 'age':
            # Exclure 'tous âges', 'tous ages', 'total'
            uniques = df_effectifs.loc[
                ~df_effectifs[col_real_name].astype(str).str.lower().isin(['tous âges', 'tous ages', 'total']),
                col_real_name
            ].dropna().unique()
        elif key_name == 'patho_effectif':
            # Exclure 'total'
            uniques = df_effectifs.loc[
                ~df_effectifs[col_real_name].astype(str).str.lower().isin(['total']),
                col_real_name
            ].dropna().unique()
        else:
            uniques = df_effectifs[col_real_name].dropna().unique()
        
        len_dict[f"len_{key_name}"] = len(uniques) + 1

# Dépenses
annees = sorted(df_depenses['annee'].dropna().unique())

pathos = sorted(df_depenses.loc[
    ~df_depenses['patho_niv1'].isin(['Total', 'Total consommants tous régimes']),
    'patho_niv1'
].dropna().unique())

postes = sorted(df_depenses['poste de dépense'].dropna().unique())
postes_niv2 = sorted(df_depenses['sous poste'].dropna().unique())

# Effectifs
depts = sorted(df_effectifs['Département'].dropna().unique())

classes_age = sorted(df_effectifs.loc[
    ~df_effectifs['Classe d\'age'].astype(str).str.lower().isin(['tous âges', 'tous ages', 'total']),
    'Classe d\'age'
].dropna().unique())

sexes = sorted(df_effectifs['Sexe'].dropna().unique())
regions = sorted(df_effectifs['Région'].dropna().unique())

pathos_eff = sorted(df_effectifs.loc[
    ~df_effectifs['patho_niv1'].astype(str).str.lower().isin(['Total', 'Total consommants tous régimes']),
    'patho_niv1'
].dropna().unique())

pprint.pprint(len_dict)

{'len_age': 21,
 'len_annee': 3,
 'len_annee_eff': 3,
 'len_departement_eff': 96,
 'len_patho_effectif': 19,
 'len_patho_niv1': 19,
 'len_poste de dépense': 5,
 'len_region': 14,
 'len_sexe': 4,
 'len_sous poste': 32}


Création d'un onglet Filtres

Cet onglet est nécessaire pour alimenter les listes de validaiton dans les filtres.

Je vais récupérer l'année, la pathologie et le poste de dépense avec la fonction UNIQUE() depuis les onglets "cleaned_data"

In [87]:
from openpyxl.utils import FORMULAE
"UNIQUE" in FORMULAE

False

# Filtres

In [88]:
wb_dep = load_workbook(output_dep)

if 'Filtres' not in wb_dep.sheetnames:
    filtres_dep = wb_dep.create_sheet('Filtres', 0)
else:
    filtres_dep = wb_dep['Filtres']
    filtres_dep.delete_rows(1, filtres_dep.max_row)

postes_list = sorted([
    str(p) for p in df_depenses['poste de dépense'].dropna().unique()
    if str(p).strip() != ""
])

annees_list = sorted([
    int(a) for a in df_depenses['annee'].dropna().unique()
    if str(a).strip() != ""
])

patho_list = sorted([
    str(p) for p in df_depenses['patho_niv1'].dropna().unique()
    if str(p).strip() != ""
])

filtres_dep['A1'] = "Poste de dépense"
for idx, val in enumerate(postes_list, start=2):
    filtres_dep[f'A{idx}'] = val

filtres_dep['B1'] = "Année"
for idx, val in enumerate(annees_list, start=2):
    filtres_dep[f'B{idx}'] = val

filtres_dep['C1'] = "Pathologie"
for idx, val in enumerate(patho_list, start=2):
    filtres_dep[f'C{idx}'] = val

filtres_dep.sheet_state = 'hidden'



## Fonctions utilitaires

Comme nous allons répéter cette opération plusieurs fois, il est préférable de créer une fonction qui effectue les mêmes opérations.

In [99]:
from openpyxl.worksheet.worksheet import Worksheet

def auto_adjust_columns(sheet: Worksheet) -> None:
    for col_idx, column in enumerate(sheet.columns, 1):
        max_length = 0
        col_letter = get_column_letter(col_idx)
        for cell in column:
            if cell.value:
                max_length = max(max_length, len(str(cell.value)))
        adjusted_width = min(max_length + 2, 50)
        sheet.column_dimensions[col_letter].width = max(adjusted_width, 12)


def add_filter_title(sheet: Worksheet, title: str, start_row: int, start_column: int) -> None:
    cell = sheet.cell(row=start_row, column=start_column)
    cell.value = title
    cell.alignment = Alignment(horizontal='center', vertical='center')
    cell.fill = PatternFill(start_color='FF1F4E78', fill_type='solid') 
    cell.font = Font(bold=True)


def add_filter_value(sheet: Worksheet, value, start_row: int, start_column: int) -> None:
    cell = sheet.cell(row=start_row, column=start_column)
    cell.value = value
    cell.alignment = Alignment(horizontal='center', vertical='center')
    cell.fill = PatternFill(start_color='FF1F4E78', fill_type='solid')
    cell.font = Font(bold=True)


def add_data_validation(sheet: Worksheet, start_row: int, start_column: int, formula: str) -> None:
    dv = DataValidation(type='list', formula1=formula)
    sheet.add_data_validation(dv)
    dv.add(sheet.cell(row=start_row, column=start_column).coordinate)


def add_filter(sheet: Worksheet, title: str, value, start_row: int, start_column: int, end_row: int, end_column: int, formula: str) -> None:
    target_val_column = end_column
    add_filter_title(sheet, title, start_row, start_column)
    add_filter_value(sheet, value, start_row, target_val_column)
    add_data_validation(sheet, start_row, target_val_column, formula)


wb_dep = load_workbook(output_dep)
wb_eff = load_workbook(output_eff)

for sheet_name in wb_dep.sheetnames:
    if sheet_name != 'Filtres':
        auto_adjust_columns(wb_dep[sheet_name])

for sheet_name in wb_eff.sheetnames:
    if sheet_name != 'Filtres':
        auto_adjust_columns(wb_eff[sheet_name])

wb_eff.save(output_eff)
wb_dep.save(output_dep)

wb_dep.close()
wb_eff.close()

Création du dashboard

Je vais créer un premier tableau de bord interactif avec des filtres.
Je vais me concentrer sur les dépenses de remboursement par pathologie et par poste de dépense.
L'utilisateur aura la possibilité de choisir l'année, la pathologie et le poste de dépense via des filtres sous forme de listes de validation.
Ces filtres permettront d'actualiser dynamiquement les formules et les graphiques.

Design :

In [100]:
COULEURS = {
    'principal': 'FF004A99',    
    'secondaire': 'FF008080',   
    'accent': 'FF4CAF50',       
    'filtre': 'FFDDEFD8',       
    'entete': 'FFEFF2F7',       
    'texte': 'FF333333',     
    'alerte': 'FFF44336',
    'filtre_bg': 'E0E0E0',      
}

sheet_dep = 'cleanedData_depenses'
sheet_eff = 'cleanedData_Effectifs'

wb_dep = load_workbook(output_dep)
wb_eff = load_workbook(output_eff)

wb_eff.save(output_eff)
wb_dep.save(output_dep)

wb_dep.close()
wb_eff.close()

In [101]:
thin = Side(border_style="thin", color="4D4D4D")
border_thin_all = Border(left=thin, right=thin, top=thin, bottom=thin)
header_fill = PatternFill(start_color='FF4472C4', fill_type='solid')
header_font = Font(bold=True, color='FFFFFF')

def apply_border_style(sheet, start_row, end_row, start_col, end_col, border_row):
    for row in sheet.iter_rows(min_row=start_row, max_row=end_row, min_col=start_col, max_col=end_col):
        for cell in row:
            if cell.row == border_row:
                cell.border = Border(bottom=thin)
            else:
                cell.border = border_thin_all

wb_dep = load_workbook(output_dep)
wb_eff = load_workbook(output_eff)

wb_eff.save(output_eff)
wb_dep.save(output_dep)

wb_dep.close()
wb_eff.close()

## Onglet postes

In [108]:
wb_dep = load_workbook(output_dep)
ws_dep = wb_dep['cleanedData_depenses']


if 'Postes' not in wb_dep.sheetnames:
    postes_sheet = wb_dep.create_sheet('Postes', 1)
else:
    postes_sheet = wb_dep['Postes']
    postes_sheet.delete_rows(1, postes_sheet.max_row)
postes_sheet.sheet_view.showGridLines = False


add_filter(
    sheet=postes_sheet,
    title='Année',
    value=int(annees_list[-1]),
    start_row=1, start_column=1,
    end_row=1, end_column=2,
    formula=f"=Filtres!$B$2:$B${len(annees_list) + 1}"
)


postes_sheet.merge_cells('A3:E5')
postes_sheet['A3'].fill = PatternFill(start_color=COULEURS['principal'], fill_type='solid')


postes_sheet['A7'] = "Total dépenses"
postes_sheet['A7'].font = Font(bold=True, size=11)
postes_sheet['B7'] = f"=IFERROR(SUMIFS('{sheet_dep}'!G:G, '{sheet_dep}'!A:A, C$1), 0)"
postes_sheet['B7'].font = Font(bold=True, size=14, color=COULEURS['principal'])
postes_sheet['B7'].number_format = '#,##0'


border_header = Border(
    left=Side(style='thin', color=COULEURS['secondaire'].replace('FF', '')),
    right=Side(style='thin', color=COULEURS['secondaire'].replace('FF', '')),
    top=Side(style='thin', color=COULEURS['secondaire'].replace('FF', '')),
    bottom=Side(style='thin', color=COULEURS['secondaire'].replace('FF', ''))
)


border_data = Border(
    left=Side(style='thin', color='CCCCCC'),
    right=Side(style='thin', color='CCCCCC'),
    top=Side(style='thin', color='CCCCCC'),
    bottom=Side(style='thin', color='CCCCCC')
)


couleur_alt_1 = PatternFill(start_color='FFF5F5F5', fill_type='solid')
couleur_alt_2 = PatternFill(start_color='FFFFFFFF', fill_type='solid')


entetes = ["Poste", "Dépenses", "Patients", "Coût/patient"]
for i, texte in enumerate(entetes):
    col = get_column_letter(i + 1)
    cell = postes_sheet[f'{col}9']
    cell.value = texte
    cell.font = Font(bold=True, color='FFFFFF', size=11)
    cell.fill = PatternFill(start_color=COULEURS['secondaire'], fill_type='solid')
    cell.alignment = Alignment(horizontal="center", vertical="center")
    cell.border = border_header


postes_sheet.row_dimensions[9].height = 22


postes_valides = [p for p in postes_list if p and "total" not in p.lower()]


for idx, poste in enumerate(postes_valides):
    row = 11 + idx
    couleur_ligne = couleur_alt_1 if idx % 2 == 0 else couleur_alt_2
   
    postes_sheet[f'A{row}'] = poste
    postes_sheet[f'A{row}'].font = Font(color=COULEURS['texte'])
    postes_sheet[f'A{row}'].fill = couleur_ligne
    postes_sheet[f'A{row}'].alignment = Alignment(horizontal="left", vertical="center")
    postes_sheet[f'A{row}'].border = border_data
   
    postes_sheet[f'B{row}'] = f"=IFERROR(SUMIFS('{sheet_dep}'!G:G, '{sheet_dep}'!A:A, C$1, '{sheet_dep}'!E:E, A{row}), 0)"
    postes_sheet[f'B{row}'].number_format = '#,##0'
    postes_sheet[f'B{row}'].font = Font(color=COULEURS['texte'])
    postes_sheet[f'B{row}'].fill = couleur_ligne
    postes_sheet[f'B{row}'].alignment = Alignment(horizontal="right", vertical="center")
    postes_sheet[f'B{row}'].border = border_data
   
    postes_sheet[f'C{row}'] = f"=IFERROR(SUMIFS('{sheet_dep}'!H:H, '{sheet_dep}'!A:A, C$1, '{sheet_dep}'!E:E, A{row}), 0)"
    postes_sheet[f'C{row}'].number_format = '#,##0'
    postes_sheet[f'C{row}'].font = Font(color=COULEURS['texte'])
    postes_sheet[f'C{row}'].fill = couleur_ligne
    postes_sheet[f'C{row}'].alignment = Alignment(horizontal="right", vertical="center")
    postes_sheet[f'C{row}'].border = border_data
   
    postes_sheet[f'D{row}'] = f"=IFERROR(B{row} / C{row}, 0)"
    postes_sheet[f'D{row}'].number_format = '#,##0.00'
    postes_sheet[f'D{row}'].font = Font(color=COULEURS['texte'])
    postes_sheet[f'D{row}'].fill = couleur_ligne
    postes_sheet[f'D{row}'].alignment = Alignment(horizontal="right", vertical="center")
    postes_sheet[f'D{row}'].border = border_data
   
    postes_sheet.row_dimensions[row].height = 18


evolution_start = 11 + len(postes_valides) + 3


postes_sheet[f'A{evolution_start}'] = "Évolution annuelle des dépenses"
postes_sheet[f'A{evolution_start}'].font = Font(bold=True, size=12, color='FFFFFF')
postes_sheet[f'A{evolution_start}'].fill = PatternFill(start_color=COULEURS['principal'], fill_type='solid')
postes_sheet.merge_cells(f'A{evolution_start}:E{evolution_start}')
postes_sheet[f'A{evolution_start}'].alignment = Alignment(horizontal="center", vertical="center")
postes_sheet.row_dimensions[evolution_start].height = 25


header_row = evolution_start + 2
entetes_evolution = ["Poste", "Dépenses 2022", "Dépenses 2023", "Variation", "Évolution %"]
for i, texte in enumerate(entetes_evolution):
    col = get_column_letter(i + 1)
    cell = postes_sheet[f'{col}{header_row}']
    cell.value = texte
    cell.font = Font(bold=True, color='FFFFFF', size=11)
    cell.fill = PatternFill(start_color=COULEURS['secondaire'], fill_type='solid')
    cell.alignment = Alignment(horizontal="center", vertical="center")
    cell.border = border_header


postes_sheet.row_dimensions[header_row].height = 22


for idx, poste in enumerate(postes_valides):
    row = evolution_start + 4 + idx
    couleur_ligne = couleur_alt_1 if idx % 2 == 0 else couleur_alt_2
   
    postes_sheet[f'A{row}'] = poste
    postes_sheet[f'A{row}'].font = Font(color=COULEURS['texte'])
    postes_sheet[f'A{row}'].fill = couleur_ligne
    postes_sheet[f'A{row}'].alignment = Alignment(horizontal="left", vertical="center")
    postes_sheet[f'A{row}'].border = border_data
   
    postes_sheet[f'B{row}'] = f"=IFERROR(SUMIFS('{sheet_dep}'!G:G, '{sheet_dep}'!A:A, 2022, '{sheet_dep}'!E:E, A{row}), 0)"
    postes_sheet[f'B{row}'].number_format = '#,##0'
    postes_sheet[f'B{row}'].font = Font(color=COULEURS['texte'])
    postes_sheet[f'B{row}'].fill = couleur_ligne
    postes_sheet[f'B{row}'].alignment = Alignment(horizontal="right", vertical="center")
    postes_sheet[f'B{row}'].border = border_data
   
    postes_sheet[f'C{row}'] = f"=IFERROR(SUMIFS('{sheet_dep}'!G:G, '{sheet_dep}'!A:A, 2023, '{sheet_dep}'!E:E, A{row}), 0)"
    postes_sheet[f'C{row}'].number_format = '#,##0'
    postes_sheet[f'C{row}'].font = Font(color=COULEURS['texte'])
    postes_sheet[f'C{row}'].fill = couleur_ligne
    postes_sheet[f'C{row}'].alignment = Alignment(horizontal="right", vertical="center")
    postes_sheet[f'C{row}'].border = border_data
   
    postes_sheet[f'D{row}'] = f"=C{row}-B{row}"
    postes_sheet[f'D{row}'].number_format = '#,##0'
    postes_sheet[f'D{row}'].font = Font(color=COULEURS['texte'])
    postes_sheet[f'D{row}'].fill = couleur_ligne
    postes_sheet[f'D{row}'].alignment = Alignment(horizontal="right", vertical="center")
    postes_sheet[f'D{row}'].border = border_data
   
    postes_sheet[f'E{row}'] = f"=IFERROR((C{row}-B{row})/B{row}, 0)"
    postes_sheet[f'E{row}'].number_format = '0.00%'
    postes_sheet[f'E{row}'].font = Font(color=COULEURS['texte'])
    postes_sheet[f'E{row}'].fill = couleur_ligne
    postes_sheet[f'E{row}'].alignment = Alignment(horizontal="right", vertical="center")
    postes_sheet[f'E{row}'].border = border_data
   
    postes_sheet.row_dimensions[row].height = 18


postes_sheet.column_dimensions['A'].width = 30
postes_sheet.column_dimensions['B'].width = 16
postes_sheet.column_dimensions['C'].width = 16
postes_sheet.column_dimensions['D'].width = 14
postes_sheet.column_dimensions['E'].width = 14


auto_adjust_columns(postes_sheet)



In [109]:
pie = PieChart()
pie.title = "Répartition des dépenses par poste"
pie.style = 10

labels = Reference(postes_sheet, min_col=1, min_row=11, max_row=10 + len(postes_valides))
data = Reference(postes_sheet, min_col=2, min_row=11, max_row=10 + len(postes_valides))

pie.add_data(data, titles_from_data=False)
pie.set_categories(labels)

# Configuration des étiquettes en pourcentage
pie.dataLabels = DataLabelList()
pie.dataLabels.showPercent = True
pie.dataLabels.showCatName = False
pie.dataLabels.showVal = False

# Nuances de vert
couleurs_vertes = ['4CAF50', '66BB6A', '81C784', 'A5D6A7', 'C8E6C9', '7CB342', '9CCC65']

series = pie.series[0]
for idx in range(len(postes_valides)):
    pt = DataPoint(idx=idx)
    couleur = couleurs_vertes[idx % len(couleurs_vertes)]
    pt.graphicalProperties.solidFill = couleur
    series.data_points.append(pt)

pie.legend.position = 'r'
postes_sheet.add_chart(pie, "G11")

wb_dep.save(output_dep)
wb_dep.close()

## Onglet depenses national

In [110]:
wb_dep = load_workbook(output_dep)
ws_dep = wb_dep['cleanedData_depenses']

if 'Dépenses' in wb_dep.sheetnames:
    del wb_dep['Dépenses']

depenses_sheet = wb_dep.create_sheet('Dépenses', 1)
depenses_sheet.sheet_view.showGridLines = False

border = Border(
    left=Side(style='thin', color='CCCCCC'),
    right=Side(style='thin', color='CCCCCC'),
    top=Side(style='thin', color='CCCCCC'),
    bottom=Side(style='thin', color='CCCCCC')
)

def style_header(cell_ref):
    cell = depenses_sheet[cell_ref]
    cell.font = Font(bold=True, color='FFFFFF')
    cell.fill = PatternFill(start_color=COULEURS['secondaire'], fill_type='solid')
    cell.alignment = Alignment(horizontal='center', vertical='center')
    cell.border = border

depenses_sheet.merge_cells('A1:H1')
depenses_sheet['A1'] = "Dépenses par pathologie"
depenses_sheet['A1'].font = Font(bold=True, size=13, color='FFFFFF')
depenses_sheet['A1'].fill = PatternFill(start_color='FF006B4F', fill_type='solid')
depenses_sheet['A1'].alignment = Alignment(horizontal='center', vertical='center')
depenses_sheet.row_dimensions[1].height = 28

add_filter(
    sheet=depenses_sheet,
    title='Poste de dépense',
    value=postes_list[0] if postes_list else '',
    start_row=3, start_column=1,
    end_row=3, end_column=2,
    formula=f"=Filtres!$A$2:$A${len(postes_list) + 1}"
)

add_filter(
    sheet=depenses_sheet,
    title='Année',
    value=int(annees_list[-1]) if annees_list else '',
    start_row=3, start_column=4,
    end_row=3, end_column=5,
    formula=f"=Filtres!$B$2:$B${len(annees_list) + 1}"
)

# Mise en forme
depenses_sheet['A3'].font = Font(bold=True, color='FFFFFF')
depenses_sheet['A3'].fill = PatternFill(start_color='FF006B4F', fill_type='solid')
depenses_sheet['D3'].font = Font(bold=True, color='FFFFFF')
depenses_sheet['D3'].fill = PatternFill(start_color='FF006B4F', fill_type='solid')

# Mise en forme
depenses_sheet['B3'].fill = PatternFill(start_color='FFFFFFFF', fill_type='solid')
depenses_sheet['B3'].font = Font(size=11, bold=True, color='FF333333')
depenses_sheet['E3'].fill = PatternFill(start_color='FFFFFFFF', fill_type='solid')
depenses_sheet['E3'].font = Font(size=11, bold=True, color='FF333333')

poste_sel = depenses_sheet['B3'].value
annee_sel = depenses_sheet['E3'].value

headers = {}
for idx, cell in enumerate(ws_dep[1], start=1):
    if cell.value is not None:
        headers[str(cell.value).strip()] = idx

col_patho = headers['pathologie'] if 'pathologie' in headers else headers.get('patho_niv1')
col_poste = headers['poste de dépense']
col_montant = headers['montant']
col_effectif = headers['Effectif']
col_annee = headers['annee'] if 'annee' in headers else None

rows = []
seen = set()

for row in ws_dep.iter_rows(min_row=2, values_only=True):
    if col_annee is not None and row[col_annee - 1] != annee_sel:
        continue
    if row[col_poste - 1] != poste_sel:
        continue

    patho = row[col_patho - 1]
    label = str(patho).strip() if patho not in (None, "") else "Non renseigné"
    
    # EXCLURE LES LIGNES DE TOTAL
    if "total" in label.lower() or "soins courants" in label.lower() or "consommateurs" in label.lower():
        continue
    if label not in seen:
        seen.add(label)
        rows.append(label)

depenses_sheet['A6'] = "Détails des pathologies"
depenses_sheet['A6'].font = Font(bold=True, size=11, color=COULEURS['principal'])

depenses_sheet['A8'] = "Pathologie"
depenses_sheet['B8'] = "Effectifs"
depenses_sheet['C8'] = "Montant (€)"
depenses_sheet['D8'] = "Coût/patient"
for h in ['A8', 'B8', 'C8', 'D8']:
    style_header(h)

start = 9
source_sheet = ws_dep.title

for i, label in enumerate(rows, start=start):
    depenses_sheet.cell(i, 1, label)
    depenses_sheet.cell(i, 2).value = (
        f'=SUMIFS({source_sheet}!$H:$H,{source_sheet}!$B:$B,$A{i},'
        f'{source_sheet}!$E:$E,$B$3,{source_sheet}!$A:$A,$E$3)'
    )
    depenses_sheet.cell(i, 3).value = (
        f'=SUMIFS({source_sheet}!$G:$G,{source_sheet}!$B:$B,$A{i},'
        f'{source_sheet}!$E:$E,$B$3,{source_sheet}!$A:$A,$E$3)'
    )
    depenses_sheet.cell(i, 4).value = f'=IFERROR(C{i}/B{i},0)'
    for c in range(1, 5):
        depenses_sheet.cell(i, c).border = border
    depenses_sheet.cell(i, 2).number_format = '#,##0'
    depenses_sheet.cell(i, 3).number_format = '#,##0'
    depenses_sheet.cell(i, 4).number_format = '#,##0.00'

# LIGNE TOTAL
last = start + len(rows) - 1 if rows else start
total_row = last + 1

depenses_sheet.cell(total_row, 1, "TOTAL")
depenses_sheet.cell(total_row, 1).font = Font(bold=True, color='FFFFFF')
depenses_sheet.cell(total_row, 1).fill = PatternFill(start_color=COULEURS['secondaire'], fill_type='solid')

depenses_sheet.cell(total_row, 2).value = f"=SUM(B{start}:B{last})"
depenses_sheet.cell(total_row, 2).font = Font(bold=True, color='FFFFFF')
depenses_sheet.cell(total_row, 2).fill = PatternFill(start_color=COULEURS['secondaire'], fill_type='solid')
depenses_sheet.cell(total_row, 2).number_format = '#,##0'

depenses_sheet.cell(total_row, 3).value = f"=SUM(C{start}:C{last})"
depenses_sheet.cell(total_row, 3).font = Font(bold=True, color='FFFFFF')
depenses_sheet.cell(total_row, 3).fill = PatternFill(start_color=COULEURS['secondaire'], fill_type='solid')
depenses_sheet.cell(total_row, 3).number_format = '#,##0'

depenses_sheet.cell(total_row, 4).value = f"=IFERROR(C{total_row}/B{total_row},0)"
depenses_sheet.cell(total_row, 4).font = Font(bold=True, color='FFFFFF')
depenses_sheet.cell(total_row, 4).fill = PatternFill(start_color=COULEURS['secondaire'], fill_type='solid')
depenses_sheet.cell(total_row, 4).number_format = '#,##0.00'


In [111]:
for c in range(1, 5):
    depenses_sheet.cell(total_row, c).border = border

if rows:
    chart = BarChart()
    chart.type = "bar"
    chart.style = 11
    chart.title = "Effectifs par pathologie"
    chart.x_axis.title = "Effectif"
    chart.y_axis.title = "Pathologie"
    chart.height = 14
    chart.width = 20

    data = Reference(depenses_sheet, min_col=2, min_row=8, max_row=last)
    cats = Reference(depenses_sheet, min_col=1, min_row=9, max_row=last)
    chart.add_data(data, titles_from_data=True)
    chart.set_categories(cats)
    
    series = chart.series[0]
    series.graphicalProperties.solidFill = "008080" 
    
    depenses_sheet.add_chart(chart, "A30")

# AGRANDIR LES COLONNES
depenses_sheet.column_dimensions['A'].width = 70
depenses_sheet.column_dimensions['B'].width = 30
depenses_sheet.column_dimensions['C'].width = 20
depenses_sheet.column_dimensions['D'].width = 20
depenses_sheet.column_dimensions['F'].width = 24
depenses_sheet.column_dimensions['G'].width = 24
depenses_sheet.column_dimensions['H'].width = 24

wb_dep.save(output_dep)
wb_dep.close()

# Filtres Effectifs

In [ ]:
wb_eff = load_workbook(output_eff)
ws_eff = wb_eff['cleanedData_Effectifs']

headers = {}
for idx, cell in enumerate(ws_eff[1], start=1):
    if cell.value is not None:
        headers[str(cell.value).strip()] = idx

# Obtenir les indices des colonnes de ws_eff
col_annee = headers.get('annee')
col_patho = headers.get('patho_niv1')
col_region = headers.get('Région')
col_dept = headers.get('Département')

# Extraire les valeurs uniques
annees_set = set()
pathos_set = set()
regions_set = set()
depts_set = set()

for row in ws_eff.iter_rows(min_row=2, values_only=True):
    if col_annee and row[col_annee - 1] is not None:
        annees_set.add(str(row[col_annee - 1]).strip())
    
    if col_patho and row[col_patho - 1] is not None:
        val = str(row[col_patho - 1]).strip()
        if val != "":
            pathos_set.add(val)
    
    if col_region and row[col_region - 1] is not None:
        val = str(row[col_region - 1]).strip()
        if val != "":
            regions_set.add(val)
    
    if col_dept and row[col_dept - 1] is not None:
        val = str(row[col_dept - 1]).strip()
        if val != "":
            depts_set.add(val)

# Trier les listes
annees_list = sorted(annees_set)
pathos_list = sorted(pathos_set)
regions_list = sorted(regions_set)
depts_list = sorted(depts_set)

if 'Filtres' in wb_eff.sheetnames:
    filtres_eff = wb_eff['Filtres']
    filtres_eff.delete_rows(1, filtres_eff.max_row)
else:
    filtres_eff = wb_eff.create_sheet('Filtres', 0)

# Remplir les colonnes :
filtres_eff['A1'] = "Année"
for idx, val in enumerate(annees_list, start=2):
    filtres_eff[f'A{idx}'] = val

filtres_eff['B1'] = "Pathologie"
for idx, val in enumerate(pathos_list, start=2):
    filtres_eff[f'B{idx}'] = val

filtres_eff['C1'] = "Région"
for idx, val in enumerate(regions_list, start=2):
    filtres_eff[f'C{idx}'] = val

filtres_eff['D1'] = "Département"
for idx, val in enumerate(depts_list, start=2):
    filtres_eff[f'D{idx}'] = val

filtres_eff.sheet_state = 'hidden'

# Sauvegarder et fermer
wb_eff.save(output_eff)
wb_eff.close()

print(f"Filtres effectifs créés:")
print(f"   - {len(annees_list)} années")
print(f"   - {len(pathos_list)} pathologies")
print(f"   - {len(regions_list)} régions")
print(f"   - {len(depts_list)} départements")

## Onglet Effectifs

In [ ]:
from collections import defaultdict

wb_eff = load_workbook(output_eff)
ws_eff = wb_eff['cleanedData_Effectifs']

if 'Effectifs' in wb_eff.sheetnames:
    del wb_eff['Effectifs']

effectifs_sheet = wb_eff.create_sheet('Effectifs', 0)
effectifs_sheet.sheet_view.showGridLines = False

border = Border(
    left=Side(style='thin', color='CCCCCC'),
    right=Side(style='thin', color='CCCCCC'),
    top=Side(style='thin', color='CCCCCC'),
    bottom=Side(style='thin', color='CCCCCC')
)

def style_header(cell_ref):
    cell = effectifs_sheet[cell_ref]
    cell.font = Font(bold=True, color='FFFFFF')
    cell.fill = PatternFill(start_color=COULEURS['secondaire'], fill_type='solid')
    cell.alignment = Alignment(horizontal='center', vertical='center')
    cell.border = border

effectifs_sheet.merge_cells('A1:H1')
effectifs_sheet['A1'] = "Diagnostic national des pathologies"
effectifs_sheet['A1'].font = Font(bold=True, size=13, color='FFFFFF')
effectifs_sheet['A1'].fill = PatternFill(start_color='FF006B4F', fill_type='solid')
effectifs_sheet['A1'].alignment = Alignment(horizontal='center', vertical='center')
effectifs_sheet.row_dimensions[1].height = 28

add_filter(
    sheet=effectifs_sheet,
    title='Année',
    value=2023,
    start_row=3,
    start_column=1,
    end_row=3,
    end_column=2,
    formula=f"=Filtres!$A$2:$A${len(annees_list) + 1}"  # ✅ dynamique
)

add_filter(
    sheet=effectifs_sheet,
    title='Région',
    value='',
    start_row=3,
    start_column=4,
    end_row=3,
    end_column=5,
    formula=f"=Filtres!$C$2:$C${len(regions_list) + 1}"  # ✅ colonne C au lieu de E
)

effectifs_sheet['A3'].font = Font(bold=True, color='FFFFFF')
effectifs_sheet['A3'].fill = PatternFill(start_color='FF006B4F', fill_type='solid')
effectifs_sheet['D3'].font = Font(bold=True, color='FFFFFF')
effectifs_sheet['D3'].fill = PatternFill(start_color='FF006B4F', fill_type='solid')

effectifs_sheet['B3'].fill = PatternFill(start_color='FFFFFFFF', fill_type='solid')
effectifs_sheet['E3'].fill = PatternFill(start_color='FFFFFFFF', fill_type='solid')

headers = {}
for idx, cell in enumerate(ws_eff[1], start=1):
    if cell.value is not None:
        headers[str(cell.value).strip()] = idx

col_annee   = headers.get('annee')
col_region  = headers.get('Région')
col_dept    = headers.get('Département')
col_patho   = headers.get('patho_niv1')
col_effectif = headers.get('Effectif')

# Vérification des colonnes
missing = [k for k, v in {
    'annee': col_annee, 'Région': col_region,
    'Département': col_dept, 'patho_niv1': col_patho,
    'Effectif': col_effectif
}.items() if v is None]
if missing:
    print(f"Colonnes manquantes : {missing}")
    print(f"Colonnes disponibles : {list(headers.keys())}")

agg_region = defaultdict(list)
agg_dept   = defaultdict(list)

for row in ws_eff.iter_rows(min_row=2, values_only=True):
    region   = row[col_region - 1]   if col_region   else None
    dept     = row[col_dept - 1]     if col_dept     else None
    patho    = row[col_patho - 1]    if col_patho    else None
    effectif = row[col_effectif - 1] if col_effectif else 0
    effectif = effectif or 0

    label_patho = str(patho) if patho else "Non renseigné"

    if region:
        agg_region[str(region)].append((label_patho, float(effectif)))
    if dept:
        agg_dept[str(dept)].append((label_patho, float(effectif)))

top_regions = sorted(agg_region.items(), key=lambda x: sum(e[1] for e in x[1]), reverse=True)[:10]
top_depts   = sorted(agg_dept.items(),   key=lambda x: sum(e[1] for e in x[1]), reverse=True)[:10]

# En-têtes tableau régions
effectifs_sheet['A6'] = "Top 10 des régions"
effectifs_sheet['A6'].font = Font(bold=True, size=11, color='FF006B4F')

effectifs_sheet['A8'] = "Région"
effectifs_sheet['B8'] = "Pathologie"
effectifs_sheet['C8'] = "Effectifs"
for h in ['A8', 'B8', 'C8']:
    style_header(h)

source_sheet = ws_eff.title
start_region = 9

for i, (region, pathos) in enumerate(top_regions):
    for j, (patho, eff) in enumerate(pathos):
        row_idx = start_region + sum(len(p[1]) for p in top_regions[:i]) + j
        effectifs_sheet.cell(row_idx, 1, region if j == 0 else "")
        effectifs_sheet.cell(row_idx, 2, patho)
        effectifs_sheet.cell(row_idx, 3).value = (
            f'=SUMIFS({source_sheet}!${chr(64+col_effectif)}:${chr(64+col_effectif)},'
            f'{source_sheet}!${chr(64+col_region)}:${chr(64+col_region)},$A{row_idx},'
            f'{source_sheet}!${chr(64+col_patho)}:${chr(64+col_patho)},$B{row_idx},'
            f'{source_sheet}!${chr(64+col_annee)}:${chr(64+col_annee)},$B$3)'
        )
        for c in range(1, 4):
            effectifs_sheet.cell(row_idx, c).border = border
        effectifs_sheet.cell(row_idx, 3).number_format = '#,##0'

last_region = start_region + sum(len(pathos) for _, pathos in top_regions) - 1 if top_regions else start_region

# En-têtes tableau départements
effectifs_sheet['E6'] = "Top 10 des départements"
effectifs_sheet['E6'].font = Font(bold=True, size=11, color='FF006B4F')

effectifs_sheet['E8'] = "Département"
effectifs_sheet['F8'] = "Pathologie"
effectifs_sheet['G8'] = "Effectifs"
for h in ['E8', 'F8', 'G8']:
    style_header(h)

start_dept = 9

for i, (dept, pathos) in enumerate(top_depts):
    for j, (patho, eff) in enumerate(pathos):
        row_idx = start_dept + sum(len(p[1]) for p in top_depts[:i]) + j
        effectifs_sheet.cell(row_idx, 5, dept if j == 0 else "")
        effectifs_sheet.cell(row_idx, 6, patho)
        effectifs_sheet.cell(row_idx, 7).value = (
            f'=SUMIFS({source_sheet}!${chr(64+col_effectif)}:${chr(64+col_effectif)},'
            f'{source_sheet}!${chr(64+col_dept)}:${chr(64+col_dept)},$E{row_idx},'
            f'{source_sheet}!${chr(64+col_patho)}:${chr(64+col_patho)},$F{row_idx},'
            f'{source_sheet}!${chr(64+col_annee)}:${chr(64+col_annee)},$B$3)'
        )
        for c in range(5, 8):
            effectifs_sheet.cell(row_idx, c).border = border
        effectifs_sheet.cell(row_idx, 7).number_format = '#,##0'

# Calcul correct de last_dept
last_dept = start_dept + sum(len(pathos) for _, pathos in top_depts) - 1 if top_depts else start_dept

# Graphique régions SANS GraphicalProperties
if top_regions:
    chart1 = BarChart()
    chart1.type = "col"
    chart1.style = 11
    chart1.title = "Effectifs par région"
    chart1.y_axis.title = "Effectifs"
    chart1.height = 13
    chart1.width = 18

    data1 = Reference(effectifs_sheet, min_col=3, min_row=8, max_row=last_region)
    cats1 = Reference(effectifs_sheet, min_col=1, min_row=9, max_row=last_region)
    chart1.add_data(data1, titles_from_data=True)
    chart1.set_categories(cats1)
    effectifs_sheet.add_chart(chart1, "A30")

# Graphique départements SANS GraphicalProperties
if top_depts:
    chart2 = BarChart()
    chart2.type = "col"
    chart2.style = 11
    chart2.title = "Effectifs par département"
    chart2.y_axis.title = "Effectifs"
    chart2.height = 13
    chart2.width = 18

    data2 = Reference(effectifs_sheet, min_col=7, min_row=8, max_row=last_dept)
    cats2 = Reference(effectifs_sheet, min_col=5, min_row=9, max_row=last_dept)
    chart2.add_data(data2, titles_from_data=True)
    chart2.set_categories(cats2)
    effectifs_sheet.add_chart(chart2, "J30")

effectifs_sheet.column_dimensions['A'].width = 50
effectifs_sheet.column_dimensions['B'].width = 50
effectifs_sheet.column_dimensions['C'].width = 20
effectifs_sheet.column_dimensions['E'].width = 50
effectifs_sheet.column_dimensions['F'].width = 50
effectifs_sheet.column_dimensions['G'].width = 20

wb_eff.save(output_eff)
wb_eff.close()

# Onglet Régional

# Onglet Départemental

## Sauvegarde finale

In [98]:
# 1. Sauvegarde des classeurs individuels (chacun sa variable)
wb_dep.save(output_dep)
wb_eff.save(output_eff)

# 2. Fermeture des classeurs (les parenthèses doivent rester totalement vides)
wb_dep.close()
wb_eff.close()